In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from nilearn.maskers import NiftiMasker

from abstract_values.utils.data import Subject, BIDS_FOLDER

In [ ]:
subject = 'pil01'
sub     = Subject(subject, bids_folder=BIDS_FOLDER)
deriv   = BIDS_FOLDER / 'derivatives' / 'encoding_models'

In [ ]:
# ── load mean CV R² images ──────────────────────────────────────────────────
def load_cvr2(model_dir):
    fn = (deriv / model_dir / f'sub-{subject}' / 'func'
          / f'sub-{subject}_task-abstractvalue_space-T1w_desc-cvr2_pe.nii.gz')
    return nib.load(str(fn))

models = {
    'vonmises':        load_cvr2('vonmises.cv'),
    'aprf':            load_cvr2('aprf.cv'),
    'aprf-shift':      load_cvr2('aprf-shift.cv'),
    'aprf-weighted':   load_cvr2('aprf-weighted.cv'),
}

model_labels = {
    'vonmises':        'VonMises\n(orientation)',
    'aprf':            'aPRF\n(value)',
    'aprf-shift':      'aPRF-shift\n(value+ses)',
    'aprf-weighted':   'aPRF-weighted\n(value)',
}
model_colors = {
    'vonmises':        '#4c72b0',
    'aprf':            '#dd8452',
    'aprf-shift':      '#55a868',
    'aprf-weighted':   '#c44e52',
}

for name, img in models.items():
    print(f'{name:16s}  dtype={img.get_data_dtype()}  shape={img.shape}')

In [ ]:
# ── ROI definitions ─────────────────────────────────────────────────────────
# (roi_label, hemi) → display name
roi_defs = [
    ('BensonV1', 'LR', 'V1'),
    ('BensonV2', 'LR', 'V2'),
    ('BensonV3', 'LR', 'V3'),
    ('NPC',      None, 'NPC (bilateral)'),
    ('NPCl',     None, 'NPCl'),
    ('NPCr',     None, 'NPCr'),
    ('NPC1',     None, 'NPC1 (bilateral)'),
    ('NPC1l',    None, 'NPC1l'),
    ('NPC1r',    None, 'NPC1r'),
    ('NPC2',     None, 'NPC2 (bilateral)'),
    ('NPC2l',    None, 'NPC2l'),
    ('NPC2r',    None, 'NPC2r'),
]

rois = {}
for roi, hemi, label in roi_defs:
    try:
        rois[label] = sub.get_roi_mask(roi, hemi=hemi)
        n = int(rois[label].get_fdata().sum())
        print(f'{label:22s}  {n:4d} voxels')
    except FileNotFoundError:
        print(f'{label:22s}  NOT FOUND')

In [ ]:
# ── extract CV R² per voxel per ROI ────────────────────────────────────────
# roi_cvr2[roi_label][model_name] = 1-D array of CV R² values
roi_cvr2 = {}

for roi_label, mask_img in rois.items():
    masker = NiftiMasker(mask_img=mask_img).fit()
    roi_cvr2[roi_label] = {
        m: masker.transform(img).squeeze()
        for m, img in models.items()
    }

print('Done.')

In [ ]:
# ── summary table ────────────────────────────────────────────────────────────
# For each ROI, restrict to voxels where AT LEAST ONE model has cvr2 > 0.
# Then compute: proportion with cvr2 > 0 and median cvr2 per model.

rows = []
model_names = list(models.keys())

for roi_label, model_vals in roi_cvr2.items():
    stack = np.column_stack([model_vals[m] for m in model_names])  # (n_vox, 3)
    any_pos = stack.max(axis=1) > 0.0        # voxels with ≥1 model cvr2 > 0
    n_total  = stack.shape[0]
    n_any    = any_pos.sum()

    for m in model_names:
        v_all = model_vals[m]          # all ROI voxels
        v_sel = v_all[any_pos]         # restricted to interesting voxels
        rows.append({
            'ROI':        roi_label,
            'model':      m,
            'n_voxels':   n_total,
            'n_any_pos':  int(n_any),
            'prop_pos_all':  (v_all > 0).mean(),           # fraction of ALL roi voxels
            'prop_pos_sel':  (v_sel > 0).mean() if n_any else np.nan,  # fraction of interesting
            'median_all':    np.median(v_all),
            'median_sel':    np.median(v_sel) if n_any else np.nan,
        })

df = pd.DataFrame(rows)
df = df.set_index(['ROI', 'model'])
print(df[['n_voxels', 'n_any_pos', 'prop_pos_all', 'prop_pos_sel', 'median_all', 'median_sel']]
      .round(3).to_string())

In [ ]:
# ── heatmap: proportion cvr2 > 0 (in interesting voxels) ────────────────────
roi_labels  = list(rois.keys())

prop_mat = np.array([
    [df.loc[(roi, m), 'prop_pos_sel'] for m in model_names]
    for roi in roi_labels
])  # (n_rois, n_models)

median_mat = np.array([
    [df.loc[(roi, m), 'median_sel'] for m in model_names]
    for roi in roi_labels
])

fig, axes = plt.subplots(1, 2, figsize=(11, 0.5 + 0.5 * len(roi_labels)))

for ax, mat, title, fmt, cmap in zip(
    axes,
    [prop_mat,   median_mat],
    ['Proportion cvr2 > 0\n(interesting voxels)', 'Median cvr2\n(interesting voxels)'],
    ['.2f',      '.3f'],
    ['YlOrRd',   'RdBu'],
):
    vmax = np.nanmax(np.abs(mat))
    if cmap == 'RdBu':
        im = ax.imshow(mat, cmap=cmap, vmin=-vmax, vmax=vmax, aspect='auto')
    else:
        im = ax.imshow(mat, cmap=cmap, vmin=0, vmax=np.nanmax(mat), aspect='auto')

    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels([model_labels[m] for m in model_names], fontsize=9)
    ax.set_yticks(range(len(roi_labels)))
    ax.set_yticklabels(roi_labels, fontsize=9)
    ax.set_title(f'sub-{subject}\n{title}', fontsize=10)

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            txt = format(v, fmt) if not np.isnan(v) else 'nan'
            ax.text(j, i, txt, ha='center', va='center', fontsize=8,
                    color='k' if cmap == 'YlOrRd' else ('w' if abs(v) > 0.6*vmax else 'k'))

    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)

plt.tight_layout()
plt.show()

In [ ]:
# ── bar chart per ROI: proportion cvr2 > 0 ──────────────────────────────────
n_rois = len(roi_labels)
n_cols = 3
n_rows = int(np.ceil(n_rois / n_cols))

fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(4.5 * n_cols, 3.5 * n_rows),
                          sharey=True)
axes = axes.flat

for ax, roi_label in zip(axes, roi_labels):
    model_vals = roi_cvr2[roi_label]
    stack = np.column_stack([model_vals[m] for m in model_names])
    any_pos = stack.max(axis=1) > 0.0
    n_any = any_pos.sum()

    props = [
        (model_vals[m][any_pos] > 0).mean() if n_any else 0.0
        for m in model_names
    ]
    colors = [model_colors[m] for m in model_names]
    bars = ax.bar(range(len(model_names)), [p * 100 for p in props],
                  color=colors, edgecolor='k', linewidth=0.7)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels([model_labels[m] for m in model_names], fontsize=8)
    ax.set_ylabel('% voxels with cvr2 > 0', fontsize=9)
    ax.set_title(f'{roi_label}  (n={stack.shape[0]}, interesting={n_any})', fontsize=9)
    ax.axhline(50, color='k', lw=0.8, linestyle='--', alpha=0.5)

for ax in list(axes)[n_rois:]:
    ax.set_visible(False)

plt.suptitle(f'sub-{subject}  —  % voxels with CV R² > 0  (in voxels where any model > 0)',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── violin plots in interesting voxels ──────────────────────────────────────
fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(4.5 * n_cols, 3.5 * n_rows),
                          sharey=False)
axes = axes.flat

for ax, roi_label in zip(axes, roi_labels):
    model_vals = roi_cvr2[roi_label]
    stack = np.column_stack([model_vals[m] for m in model_names])
    any_pos = stack.max(axis=1) > 0.0

    data_list = [model_vals[m][any_pos] for m in model_names]

    if any_pos.sum() >= 2:
        parts = ax.violinplot(data_list, positions=range(len(model_names)),
                              showmedians=True, showextrema=False)
        for body, m in zip(parts['bodies'], model_names):
            body.set_facecolor(model_colors[m])
            body.set_alpha(0.7)
        parts['cmedians'].set_color('k')

    ax.axhline(0, color='k', lw=0.8, linestyle='--', alpha=0.5)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels([model_labels[m] for m in model_names], fontsize=8)
    ax.set_ylabel('CV R²', fontsize=9)
    ax.set_title(f'{roi_label}  (n_sel={any_pos.sum()})', fontsize=9)

for ax in list(axes)[n_rois:]:
    ax.set_visible(False)

plt.suptitle(f'sub-{subject}  —  CV R² distribution in interesting voxels (any model > 0)',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()